# BENZI — LoRA fine-tune on Colab (~2–3 hours)

**Before you start:** *Runtime → Change runtime type → **T4 GPU***

**Every run:** *Runtime → **Disconnect and delete runtime***, then open this notebook from GitHub and *Run all*.

| Step | What | Time |
|------|------|------|
| Setup | Clone + install | ~3 min |
| Dataset | Empathy JSONL | ~5 min |
| Train | Qwen2.5-3B LoRA | ~45–90 min |
| Merge + zip | Download for Mac | ~15 min |

After download see `benzi-server/docs/COLAB_TRAIN.md`.


In [ ]:
# GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable T4 GPU: Runtime → Change runtime type → GPU"
print("CUDA OK")


In [ ]:
# Fresh clone (absolute path — never use %cd final-year-benzi/...)
import os
import shutil
import subprocess
from pathlib import Path

CONTENT = Path("/content")
REPO_DIR = CONTENT / "final-year-benzi"
ML_DIR = REPO_DIR / "fyp-ml-demos"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    print("Removed old clone:", REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/sameedsaeed123/final-year-benzi.git", str(REPO_DIR)],
    check=True,
)
os.chdir(ML_DIR)

# Patch stale scripts if Colab cached an old shallow clone
train_py = ML_DIR / "finetune" / "train_qlora.py"
text = train_py.read_text(encoding="utf-8")
if "use_mps_device" in text:
    train_py.write_text(
        text.replace("        use_mps_device=False,\n", "").replace("        use_mps_device=False,\r\n", ""),
        encoding="utf-8",
    )
    print("Patched train_qlora.py (removed use_mps_device)")

assert (ML_DIR / "finetune" / "train_qlora.py").is_file()
assert "use_mps_device" not in train_py.read_text(encoding="utf-8"), "train_qlora still broken — pull latest from GitHub"
print("Working directory:", ML_DIR.resolve())


In [ ]:
# Install deps (Colab: keep numpy 2.x — do not use requirements-finetune.txt)
import subprocess, sys
from pathlib import Path

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("-r", "requirements.txt")
colab = Path("requirements-finetune-colab.txt")
if colab.is_file():
    pip("-r", str(colab))
else:
    pip("datasets>=2.19.0", "peft>=0.11.0", "accelerate>=0.30.0",
        "sentencepiece>=0.2.0", "protobuf>=4.25.0", "tqdm>=4.66.0")
pip("bitsandbytes>=0.43.0", "accelerate")

import numpy as np, bitsandbytes as bnb, torch
print("numpy", np.__version__, "| bitsandbytes", bnb.__version__)


In [ ]:
# Step 1 — dataset (~5 min)
!python finetune/prepare_dataset.py --max-total 1200


In [ ]:
# Step 2 — train (~45–90 min). Wait until you see: Saved LoRA adapter
!python finetune/train_qlora.py --model Qwen/Qwen2.5-3B-Instruct --max-steps 60 --max-length 384 --batch-size 1 --grad-accum 4


**Faster option (~20 min):** comment out Step 2 above and run:
```python
!python finetune/train_qlora.py --benzi-lite
```


In [ ]:
# Step 3 — merge (~10 min) — only after Step 2 succeeds
from pathlib import Path
adapter_cfg = Path("finetune/adapters/benzi-lora/adapter_config.json")
if not adapter_cfg.is_file():
    raise RuntimeError(
        "Training did not finish — no adapter_config.json.\n"
        "Re-run Step 2 and wait for 'Saved LoRA adapter' before merge."
    )
!python finetune/merge_lora.py --model Qwen/Qwen2.5-3B-Instruct


In [ ]:
# Step 4 — download zip
import shutil
from pathlib import Path
from google.colab import files

merged = Path("finetune/merged/benzi-empathetic-hf")
if not merged.is_dir():
    raise RuntimeError("Merged model missing — complete Step 3 first.")

shutil.make_archive("/content/benzi-empathetic-trained", "zip", merged)
files.download("/content/benzi-empathetic-trained.zip")
print("Download started: benzi-empathetic-trained.zip")


## On your Mac

```bash
mkdir -p ~/benzi-models && cd ~/benzi-models
unzip ~/Downloads/benzi-empathetic-trained.zip -d benzi-empathetic-hf
cd benzi-empathetic-hf
cat > Modelfile << 'EOF'
FROM .
PARAMETER temperature 0.65
PARAMETER num_ctx 4096
SYSTEM You are BENZI AI — supportive wellness between therapy sessions. Not a therapist. Defer clinical questions to their therapist.
EOF
ollama create benzi-empathetic-trained -f Modelfile
```

In `benzi-server/.env`: `OLLAMA_MODEL=benzi-empathetic-trained` then restart the API.
